In [ ]:
# @title Cell 1: Update & Install
!pip install -U -q yfinance tensorflow scikit-learn nltk groq

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Deep Learning Imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

# NLP & Sentiment Imports
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("✅ Setup, DL, NLP & Groq Libraries Complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 23.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-

In [ ]:
# @title Cell 2: Configuration & API
import os
from groq import Groq

# --- USER INPUTS ---
SHARE_NAME = input("Enter Share Ticker (e.g., TCS.NS, NVDA, AAPL): ").strip().upper()
try:
    SHARES_OWNED = float(input("Enter number of shares owned: "))
except ValueError:
    SHARES_OWNED = 0
    print("Invalid number, defaulting to 0.")

# --- GROQ API SETUP ---
# Paste your new Groq key here
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm_available = False
try:
    client = Groq()
    llm_available = True
    print(f"✅ Groq API Connected. Analyzing {SHARE_NAME}...")
except Exception as e:
    print(f"⚠️ API Error: {e}")

Enter Share Ticker (e.g., TCS.NS, NVDA, AAPL): ORCL
Enter number of shares owned: 0
✅ Groq API Connected. Analyzing ORCL...


In [ ]:
# @title Cell 3: Fetch Data & Sentiment Analysis
def get_clean_stock_data(ticker):
    print(f"Fetching 5-year data for {ticker}...")
    df = yf.download(ticker, period="5y", interval="1d", progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.reset_index(inplace=True)

    if 'Date' not in df.columns and 'index' in df.columns:
        df.rename(columns={'index': 'Date'}, inplace=True)

    df['Date'] = pd.to_datetime(df['Date'])
    df.sort_values(by='Date', ascending=True, inplace=True)
    df.dropna(subset=['Close'], inplace=True)
    return df.reset_index(drop=True)

df = get_clean_stock_data(SHARE_NAME)

# --- NEW: SENTIMENT ANALYSIS ---
print("Fetching latest news headlines...")
ticker_obj = yf.Ticker(SHARE_NAME)
news_data = ticker_obj.news
sia = SentimentIntensityAnalyzer()

today_sentiment = 0.0
print("\n--- Recent Headlines ---")
if news_data:
    scores = []
    # Analyze the top 5 recent articles
    for article in news_data[:5]:
        title = article.get('title', '')
        score = sia.polarity_scores(title)['compound'] # Returns -1 to 1
        scores.append(score)
        print(f"[{score:+.2f}] {title}")

    today_sentiment = sum(scores) / len(scores) if scores else 0.0
    print(f"\n=> Overall Live Sentiment Score: {today_sentiment:+.2f}")
else:
    print("No recent news found. Defaulting to neutral (0.0).")

# --- FEATURE ENGINEERING ---
df['MA_10'] = df['Close'].rolling(window=10).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()

delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))
df['Return'] = df['Close'].pct_change()

# Initialize Sentiment column (0.0 for history, real score for today)
df['Sentiment'] = 0.0
df.dropna(inplace=True)

# Apply the live sentiment score to the very last row for today's prediction
df.iloc[-1, df.columns.get_loc('Sentiment')] = today_sentiment

# Target: 1 if price in 5 days is higher than today, else 0
df['Target'] = (df['Close'].shift(-5) > df['Close']).astype(int)

print(f"\n✅ Data Ready. Rows: {len(df)}")

Fetching 5-year data for ORCL...
Fetching latest news headlines...

--- Recent Headlines ---
[+0.00] 
[+0.00] 
[+0.00] 
[+0.00] 
[+0.00] 

=> Overall Live Sentiment Score: +0.00

✅ Data Ready. Rows: 1207


In [ ]:
# @title Cell 4: LSTM Preprocessing (3D Tensors)
# Now utilizing 6 Features!
features = ['Close', 'MA_10', 'MA_50', 'RSI', 'Return', 'Sentiment']
SEQUENCE_LENGTH = 30 # 30-day lookback window

# 1. Scale the Data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_features = scaler.fit_transform(df[features])

# 2. Create Sequences (Sliding Window)
X, y = [], []
targets = df['Target'].values

for i in range(SEQUENCE_LENGTH, len(scaled_features) - 5):
    X.append(scaled_features[i-SEQUENCE_LENGTH:i])
    y.append(targets[i])

X, y = np.array(X), np.array(y)

# 3. Time-Series Train/Test Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"✅ Sequences Created with 6 Features (Including Sentiment).")
print(f"Training Data Shape: {X_train.shape}")

✅ Sequences Created with 6 Features (Including Sentiment).
Training Data Shape: (937, 30, 6)


In [ ]:
# @title Cell 5: Train LSTM Model
# 1. Build LSTM Architecture
model = Sequential()

# First LSTM layer with Dropout to prevent overfitting
model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.2))

# Second LSTM layer
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))

# Output layer (Sigmoid for binary classification: 0 to 1 probability)
model.add(Dense(units=1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training LSTM Model (This may take a minute)...")
# Train the model (Using small epochs/batch_size for fast Colab execution)
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test), verbose=0)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

# 2. Predict "Tomorrow" using the most recent 30 days of data
latest_30_days = scaled_features[-SEQUENCE_LENGTH:]
# Reshape for the model: (1 sample, 30 time steps, 5 features)
latest_30_days_tensor = np.array([latest_30_days])

buy_prob = model.predict(latest_30_days_tensor, verbose=0)[0][0]
current_price = df['Close'].values[-1]

# 3. Logic & Decision
if buy_prob > 0.55:
    market_signal = "BULLISH (Uptrend)"
    action = "BUY"
elif buy_prob < 0.45:
    market_signal = "BEARISH (Downtrend)"
    action = "SELL"
else:
    market_signal = "NEUTRAL"
    action = "HOLD"

final_advice = action
if action == "SELL" and SHARES_OWNED == 0:
    final_advice = "AVOID"

print(f"\n✅ Training Complete.")
print(f"LSTM Test Accuracy: {test_acc:.2f}")
print(f"Prediction Confidence (Up): {buy_prob:.2f}")
print(f"Signal: {market_signal}")
print(f"✅ FINAL ADVICE: {final_advice}")

Training LSTM Model (This may take a minute)...

✅ Training Complete.
LSTM Test Accuracy: 0.52
Prediction Confidence (Up): 0.46
Signal: NEUTRAL
✅ FINAL ADVICE: HOLD


In [ ]:
# @title Cell 6: AI Reasoning
print("--- AI Analysis ---\n")

if llm_available:
    prompt = f"""
    You are a senior financial analyst.
    I am looking at the stock '{SHARE_NAME}'.

    1. Deep Learning Prediction: Our LSTM model predicts a '{market_signal}' 5-day trend.
    2. User Situation: The user currently owns {SHARES_OWNED} shares.
    3. Algorithm Recommendation: '{final_advice}'.
    4. Current Price: {current_price:.2f}.

    Please provide a short explanation (3-4 bullet points) for this recommendation.
    Focus on:
    - Why this advice suits the user's current portfolio situation.
    - General sector performance or recent news typical for this company.
    - A disclaimer that this is AI generated based on mathematical momentum.
    """

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama-3.1-8b-instant",
        )
        print("="*40)
        print(chat_completion.choices[0].message.content)
        print("="*40)

    except Exception as e:
        print(f"❌ AI Analysis Failed: {e}")
else:
    print("API not configured. Skipping reasoning.")

--- AI Analysis ---

**Recommendation for ORCL: HOLD**

Based on our analysis, we recommend a HOLD position for ORCL. Here are the key reasons for this recommendation:

• **Suitable for a new investor**: With no shares currently held, a HOLD recommendation allows you to avoid any potential market fluctuations and enables you to further monitor Oracle's performance without an existing position. This approach helps you gather data and make an informed decision when to enter the market.

• **Sector performance**: The technology sector has experienced fluctuations in recent years, with some companies performing better than others. Oracle, as a leader in enterprise software, has continued to report stable earnings, indicating its resilience during market downturns.

• **Recent news and momentum**: While there may be recent news or trends impacting Oracle's stock, our algorithm's prediction of NEUTRAL momentum over the next 5 days is based on historical data and doesn't account for any futur